# NeuroWrite-PT — Classificação de Caracteres Manuscritos
**Universidade Cruzeiro do Sul | Ciência de Dados e Aprendizagem de Máquina | 2026**

**Integrantes:** Carlos Eduardo Franco Gabriel; Ian de Castro Carraro; Isadora Baena Lourenço; Miguel Ferreira Palmieri Souza Rocha.

**Experimento:** comparação de SVM/RBF e MLP na classificação de caracteres manuscritos isolados, incluindo a base própria da Isadora e um referencial independente com EMNIST Letters.

**Como executar no Google Colab:** em **Ambiente de execução → Executar tudo**. A primeira célula baixa automaticamente os arquivos de dados e gráficos do repositório GitHub usando `curl`. **Não exige login no Google Drive, ZIP ou dados Base64 no código.** Os arquivos aparecem nas pastas `data/` e `assets/` da aba Arquivos do Colab. É necessária conexão com a internet **somente para o download inicial**, além das bibliotecas Python usuais.

**Apresentação:** na versão previamente executada, os gráficos e resultados já estão visíveis sem executar as células. Há também uma versão **não executada** para demonstrar o treinamento desde o início.


## Resumo
Este projeto compara dois algoritmos supervisionados para classificação de caracteres manuscritos isolados. Foram realizados dois experimentos distintos: (i) comparação de SVM e MLP em 26 classes A–Z do EMNIST Letters, com os modelos e métricas registrados na GitHub Action; (ii) avaliação local com caracteres manuscritos de uma escritora, utilizando 1.063 imagens candidatas aprovadas na triagem, distribuídas por dez folhas e 37 classes. A representação de cada caractere usa 28 × 28 pixels normalizados. Para reduzir o vazamento entre exemplos semelhantes, a avaliação local separa fotos inteiras em treinamento e teste e utiliza validação cruzada agrupada por foto no conjunto de treino. São reportados acurácia, precisão, recall, F1 macro, matrizes de confusão e exemplos de confusões. As conclusões diferenciam as limitações do conjunto público e da coleta própria: apenas uma escritora estava disponível e os recortes foram gerados por um procedimento semiautomatizado. Portanto, os resultados não demonstram generalização entre escritores nem reconhecimento de texto cursivo.

## Introdução
O reconhecimento de letras manuscritas é uma tarefa de **classificação multiclasse supervisionada**: um vetor numérico de pixels é associado a uma classe conhecida. A variação individual da escrita, condições de fotografia, espessura do traço e presença de acentos dificultam a generalização.

O projeto iniciou com o **EMNIST Letters** (base pública do NIST, 26 classes A–Z, com maiúsculas/minúsculas agrupadas), para estabelecer um referencial controlado. Em seguida, foi preparada uma base própria a partir de folhas de caligrafia da Isadora. Os experimentos são **separados**, porque têm classes, proveniência e pré-processamento diferentes.

## Objetivo
Comparar uma **SVM com kernel RBF** e uma **MLP com uma camada oculta** na classificação de caracteres manuscritos isolados. Analisar métricas de desempenho e erros, demonstrando a diferença entre uma base pública padronizada e uma base real de escrita individual. A hipótese é que algoritmos diferentes apresentem padrões distintos de erro, e que dados com diversidade limitada restrinjam a generalização.

## Desenvolvimento — Parte 1: dados e preparação
**Base própria:** fotografias de dez folhas de uma única participante, com três blocos de caracteres por folha. Foram encontrados 1.109 recortes candidatos; a triagem visual assistida manteve 1.063 e excluiu 46. O dataset está num **arquivo real** `data/Isadora_Xy_1063_37_classes.npz`, contendo vetores de 784 pixels no intervalo [0,1], rótulos, IDs e fotos de origem. Há **37 classes com exemplos efetivos**.

O processamento prévio incluiu identificação das linhas de caderno, recorte por posição, isolamento do traço, centralização e redimensionamento para 28 × 28. Esse procedimento não é um leitor universal de qualquer documento.

**Protocolo de avaliação:** duas fotos inteiras são reservadas como teste final; as demais são usadas para ajuste, com três folds de validação cruzada por foto (**GroupKFold**). Nenhum exemplo de uma mesma foto pertence simultaneamente ao treino e ao teste. O grupo de validação contém fotos diferentes, mas pertence à mesma escritora.


### Arquivos necessários e execução automática

Os arquivos estão publicados na pasta [`presentation/` do NeuroWrite-PT](https://github.com/Carlosfrancog/NeuroWrite-PT/tree/feat/apresentacao-reproducivel-isadora/presentation). A célula 00 os baixa por **HTTPS com `curl`** e valida a integridade com **SHA-256**. O código não contém o dataset serializado, não solicita autorização do Google Drive e não precisa manipular ZIP.

```text
/content/
├── data/
│   └── Isadora_Xy_1063_37_classes.npz
├── assets/
│   ├── metricas.csv
│   ├── comparacao.png
│   ├── matriz_confusao_svm.png
│   ├── matriz_confusao_mlp.png
│   ├── distribuicao.png
│   ├── amostras.png
│   └── resumo_diagnostico.csv
└── resultados/          # gerada durante o treinamento
```

A revisão Git usada para baixar os recursos está fixada, de modo que a reexecução use os mesmos arquivos validados. O notebook **não precisa estar salvo no repositório** para funcionar: basta abri-lo no Colab e selecionar **Executar tudo**.


In [ ]:
# 00 — BAIXAR DADOS E GRÁFICOS DO GITHUB (SEM GOOGLE DRIVE OU ZIP).
# Os recursos são ARQUIVOS REAIS baixados com curl, nunca Base64 dentro do código.
from pathlib import Path
from hashlib import sha256
import subprocess
import os

URL_GITHUB = 'https://raw.githubusercontent.com/Carlosfrancog/NeuroWrite-PT/b583d29bc559e090163a3374d7d46480ed15863e/presentation'
BASE_ARQUIVOS = Path.cwd()  # /content no Google Colab
SHA256_ARQUIVOS = {'assets/amostras.png': '322e8d3d872f60eed7b35b2400c75cd698bfa9234ef4a41bd67f283630e846f7', 'assets/comparacao.png': '879767d1ec5c6940858d801b37230f84aaa356d3c343f94373297be46b581730', 'assets/distribuicao.png': '43433941a4c210a5cb488d2545516dd8607d2b999a56dfb066ca1131625c57e1', 'assets/matriz_confusao_mlp.png': 'f8723ef61cad119a22eacb4eba7a0538a99b25e729d59fc86d2fb65b4ff7f9c4', 'assets/matriz_confusao_svm.png': '2bd8fdb95244c15fb59e1eed5834437b3d51950c36c17a5f3c9631a76c3981d9', 'assets/metricas.csv': 'ca33aef13325dbd548f29c7346a12419abd00fec55c82cfa4cf24d4eb76038df', 'assets/resumo_diagnostico.csv': '9979ad0170a7929cdad428b7110c9df8a7387e13d2c2c33993251d1c79a9dcac', 'data/Isadora_Xy_1063_37_classes.npz': '17f66cdf1a30f482c6d7d3e1d555e89639a1a31a8de335d2125d2ac193047b3f'}

def garantir_recursos(nomes=None):
    """Baixa arquivos ausentes/alterados por curl e valida SHA-256."""
    nomes = list(SHA256_ARQUIVOS) if nomes is None else list(nomes)
    for relativo in nomes:
        esperado = SHA256_ARQUIVOS[relativo]
        destino = BASE_ARQUIVOS / relativo
        destino.parent.mkdir(parents=True, exist_ok=True)
        if destino.is_file() and sha256(destino.read_bytes()).hexdigest() == esperado:
            print('OK', relativo)
            continue
        temporario = destino.with_name(destino.name + '.download')
        url = URL_GITHUB + '/' + relativo
        print('Baixando do GitHub:', relativo)
        try:
            subprocess.run(['curl', '--fail', '--location', '--show-error', '--silent',
                            '--retry', '3', '--connect-timeout', '15',
                            '--output', str(temporario), url], check=True)
            obtido = sha256(temporario.read_bytes()).hexdigest()
            if obtido != esperado:
                raise RuntimeError(f'Arquivo {relativo} com SHA-256 divergente; download recusado')
            os.replace(temporario, destino)
        finally:
            if temporario.exists():
                temporario.unlink()
    print('Recursos disponíveis na aba Arquivos:',
          ', '.join(sorted({p.split('/')[0] for p in nomes})))

garantir_recursos()


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay, confusion_matrix)

SEED = 42
LETRAS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
# Os dados são carregados de um arquivo .npz real na aba Arquivos do Colab.
p = np.load(Path('data/Isadora_Xy_1063_37_classes.npz'), allow_pickle=False)
# Opcional: somente exporta figuras se a celula for executada; cria o destino sozinha.
OUT = Path('resultados')
OUT.mkdir(exist_ok=True)
X = p["X"].astype(np.float32)
y = p["y"]
groups = p["fontes"]
ids = p["ids"]
assert X.shape == (1063, 784) and len(y) == len(groups) == len(ids)
assert 0 <= X.min() and X.max() <= 1
print("IMAGENS:", X.shape[0], "| ATRIBUTOS:", X.shape[1], "| CLASSES:", len(np.unique(y)))
print("FOTOS DE ORIGEM:", len(np.unique(groups)))
counts = pd.Series(y).value_counts().sort_index()
display(pd.DataFrame({"classe":counts.index,"quantidade":counts.values}).T)
fig, ax = plt.subplots(figsize=(13,3.8))
ax.bar(counts.index, counts.values, color="#2f638b")
ax.set(title="Distribuição dos caracteres — Isadora", ylabel="Amostras", xlabel="Classe")
ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.savefig(OUT/"distribuicao_isadora.png", dpi=140); plt.show()

### Exemplos de imagens normalizadas
Cada quadrado representa um pixel de intensidade padronizada. A ordem das amostras demonstrativas é determinada pela semente fixa; exemplos são retirados da base original sem edição adicional.

In [ ]:
rng = np.random.default_rng(SEED)
exemplos = rng.choice(len(X), size=18, replace=False)
fig, axs = plt.subplots(3, 6, figsize=(12, 6))
for ax, idx in zip(axs.flat, exemplos):
    ax.imshow(X[idx].reshape(28,28), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{y[idx]}  |  {str(groups[idx]).replace('.jpg','')}", fontsize=8)
    ax.axis("off")
fig.suptitle("Amostras reais da base própria após segmentação", fontsize=13)
fig.tight_layout()
fig.savefig(OUT/"amostras_isadora.png", dpi=150)
plt.show()

## Desenvolvimento — Parte 2: algoritmos e validação
**SVM RBF:** classificador de margens com kernel radial, capaz de construir fronteiras não lineares. O escalonador `StandardScaler` é ajustado somente nos dados de treino de cada fold.

**MLP:** rede neural com 784 entradas, 128 unidades ocultas e uma saída por classe, treinada por propagação direta e retropropagação do gradiente. A configuração `early_stopping=False` indica que **não** houve parada antecipada; o limite de épocas foi fixado em `max_iter=130`, podendo haver risco de sobreajuste. A topologia acompanha o número de classes efetivamente presentes neste experimento.

Os hiperparâmetros são **fixados antes do teste** para preservar a comparação: SVM RBF com `C=10`; MLP `(128,)`, `max_iter=130`, `random_state=42`. Foi aplicada validação cruzada por foto, com F1 macro como critério principal. Essas configurações não constituem busca exaustiva de hiperparâmetros.

In [ ]:
separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train, idx_test = next(separador.split(X, y, groups=groups))
Xt, Xv = X[idx_train], X[idx_test]
yt, yv = y[idx_train], y[idx_test]
folders_train = sorted(set(groups[idx_train])); folders_test = sorted(set(groups[idx_test]))
assert not (set(folders_train) & set(folders_test))
assert set(yv).issubset(set(yt))
print("Treino:", len(yt), "amostras,", len(folders_train), "fotos")
print("Teste:", len(yv), "amostras,", len(folders_test), "fotos")
print("Fotos de teste:", ", ".join(folders_test))
modelos = {
    "SVM": Pipeline([("escala", StandardScaler()),
                     ("modelo", SVC(kernel="rbf", C=10, gamma="scale"))]),
    "MLP": Pipeline([("escala", StandardScaler()),
                     ("modelo", MLPClassifier(hidden_layer_sizes=(128,), max_iter=130,
                                               early_stopping=False, n_iter_no_change=12,
                                               random_state=SEED))])
}
cv = GroupKFold(n_splits=3)
resultados = []
predicoes = {}
for nome, pipe in modelos.items():
    print("Calculando validação e treinando:", nome)
    scores = cross_val_score(pipe, Xt, yt, groups=groups[idx_train],
                             cv=cv, scoring="f1_macro", n_jobs=1)
    pipe.fit(Xt, yt)
    previsto = pipe.predict(Xv)
    predicoes[nome] = previsto
    linha = {"modelo":nome,"amostras_treino":len(yt),"amostras_teste":len(yv),
             "f1_macro_cv_media":float(scores.mean()),
             "f1_macro_cv_desvio":float(scores.std()),
             "acuracia_teste":float(accuracy_score(yv, previsto)),
             "precisao_macro_teste":float(precision_score(yv, previsto, average="macro", zero_division=0)),
             "recall_macro_teste":float(recall_score(yv, previsto, average="macro", zero_division=0)),
             "f1_macro_teste":float(f1_score(yv, previsto, average="macro", zero_division=0))}
    resultados.append(linha)
    print(nome, "F1 CV:", np.round(scores, 3), "| F1 teste:", f"{linha['f1_macro_teste']:.3f}")
resultados = pd.DataFrame(resultados).sort_values('f1_macro_teste', ascending=False)
resultados.to_csv(OUT/"metricas_isadora.csv", index=False)
display(resultados.round(4))
json.dump({"origem":"folhas manuscritas Isadora","seed":SEED,
           "fotos_treino":folders_train,"fotos_teste":folders_test,
           "treino":len(yt),"teste":len(yv),
           "classes":sorted(np.unique(y).tolist())},
          (OUT/"metodologia.json").open('w', encoding='utf-8'),
          indent=2, ensure_ascii=False)

## Desenvolvimento — Parte 3: resultados e discussão
São apresentadas as métricas do conjunto de teste formado por **fotos reservadas**. A seleção de fotos, a semente e os hiperparâmetros estão explícitos na seção anterior. Os resultados devem ser interpretados como avaliação **dentro da caligrafia de uma única autora**, não como desempenho populacional.

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,4))
for _, linha in resultados.iterrows():
    axs[0].bar(linha['modelo'], linha['acuracia_teste'], color="#3b6e9b")
    axs[1].bar(linha['modelo'], linha['f1_macro_teste'], color="#478e77")
for ax, ttl in zip(axs, ["Acurácia — teste isolado", "F1 macro — teste isolado"]):
    ax.set_ylim(0,1); ax.set_title(ttl); ax.grid(axis="y",alpha=.2)
    for bar in ax.patches: ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.015,
                                  f"{bar.get_height():.1%}",ha="center",fontsize=10)
fig.tight_layout(); fig.savefig(OUT/"comparacao_isadora.png",dpi=160);plt.show()
nomes_classes = sorted(np.unique(y).tolist())
for nome, previsto in predicoes.items():
    fig, ax = plt.subplots(figsize=(12,10))
    ConfusionMatrixDisplay.from_predictions(yv, previsto,
        labels=nomes_classes, display_labels=nomes_classes,
        cmap="Blues", xticks_rotation=90, values_format="d", ax=ax,
        colorbar=False)
    ax.set_title(f"Matriz de confusão — {nome} (teste por foto)")
    fig.tight_layout(); fig.savefig(OUT/f"matriz_confusao_isadora_{nome.lower()}.png",dpi=135)
    plt.show()
    print(f"RELATÓRIO DE CLASSIFICAÇÃO — {nome}")
    print(classification_report(yv, previsto, labels=nomes_classes, zero_division=0))
for nome, previsto in predicoes.items():
    erros = pd.DataFrame({"id":ids[idx_test],"fonte":groups[idx_test],
                         "esperado":yv,"previsto":previsto})
    erros = erros[erros["esperado"]!=erros["previsto"]]
    erros.to_csv(OUT/f"erros_isadora_{nome.lower()}.csv",index=False)
    print(nome, "erros no teste:",len(erros),"/",len(yv))

### Referencial EMNIST: experimento público independente
Para permitir contraste metodológico, são reproduzidas abaixo as métricas **já medidas** na GitHub Action bem-sucedida [#12 / execução 42309016187](https://github.com/Carlosfrancog/NeuroWrite-PT/actions/runs/42309016187). A base pública contém 2.600 amostras de treinamento e 520 amostras de teste, 26 categorias A–Z. A busca de hiperparâmetros ocorreu exclusivamente no treino, com `StratifiedKFold(3)`; os resultados foram lidos do CSV original da Action e não recomputados com a base da Isadora. Portanto, **não comparar os percentuais como um duelo diretamente controlado**, pois as fontes, partições e classes são diferentes.

In [ ]:
# Referencial EMNIST: imagens e métricas do experimento público, obtidas via GitHub.
# Esta célula funciona até isoladamente, sem executar as anteriores.
from pathlib import Path
from hashlib import sha256
from IPython.display import display, Image as IPyImage
import subprocess
import pandas as pd

_BASE_URL = 'https://raw.githubusercontent.com/Carlosfrancog/NeuroWrite-PT/b583d29bc559e090163a3374d7d46480ed15863e/presentation'
_REF_HASHES = {'assets/metricas.csv': 'ca33aef13325dbd548f29c7346a12419abd00fec55c82cfa4cf24d4eb76038df', 'assets/comparacao.png': '879767d1ec5c6940858d801b37230f84aaa356d3c343f94373297be46b581730', 'assets/matriz_confusao_svm.png': '2bd8fdb95244c15fb59e1eed5834437b3d51950c36c17a5f3c9631a76c3981d9', 'assets/matriz_confusao_mlp.png': 'f8723ef61cad119a22eacb4eba7a0538a99b25e729d59fc86d2fb65b4ff7f9c4'}
if 'garantir_recursos' in globals():
    garantir_recursos(_REF_HASHES)
else:
    for rel, correto in _REF_HASHES.items():
        caminho = Path(rel)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        if not caminho.is_file() or sha256(caminho.read_bytes()).hexdigest() != correto:
            subprocess.run(['curl', '-fLsS', '--retry', '3', '--output', str(caminho),
                            _BASE_URL + '/' + rel], check=True)
        if sha256(caminho.read_bytes()).hexdigest() != correto:
            raise RuntimeError('Integridade inválida: ' + rel)

publico = pd.read_csv('assets/metricas.csv')
display(publico.round(4))
print('Comparação EMNIST — GitHub Action:')
display(IPyImage(filename='assets/comparacao.png', width=640))
print('Matrizes de confusão originais:')
display(IPyImage(filename='assets/matriz_confusao_svm.png', width=520))
display(IPyImage(filename='assets/matriz_confusao_mlp.png', width=520))


### Limitações e validade
1. A base própria tem apenas **uma escritora**; a separação por foto reduz dependência entre imagens da mesma página, mas não testa mudança de pessoa.
2. Os dados próprios são provenientes de fotografias reais, com correções semiautomáticas e triagem assistida; ainda pode haver ruído de rótulo/segmentação.
3. O EMNIST Letters agrega diferentes caixas tipográficas na mesma classe e não representa exatamente a coleta manuscrita própria.
4. São classificadas **letras isoladas**; reconhecimento de palavras, cursiva e frases depende de segmentação e outras técnicas não avaliadas aqui.
5. A amostra própria não deve ser descrita como um benchmark amplo; os resultados são exploratórios, com apenas dez fotos de uma única pessoa.
6. A confiança emitida por uma MLP não é garantia de acerto. Escores da SVM sem calibração não são probabilidades.

## Conclusão
Neste experimento com a base manuscrita da Isadora, a **SVM RBF** obteve a melhor avaliação nas duas fotografias reservadas: **60.09% de acurácia e 0.5776 de F1 macro**. A **MLP** atingiu **52.11% de acurácia e 0.5132 de F1 macro**. Na validação cruzada agrupada por fotografia, as médias de F1 macro foram, respectivamente, **0.4648** e **0.4206**. Os testes usam 850 letras no treino e 213 na avaliação final, oriundas de fotos diferentes, mas da **mesma escritora**.

No experimento independente EMNIST, a SVM também foi superior à MLP (acurácias **73.65%** e **66.35%**, respectivamente). As duas bases não são diretamente comparáveis por conterem diferentes classes, imagens e protocolos.

O projeto demonstra a aplicação de classificação supervisionada de letras isoladas, com controle de origem das amostras e análise de erros. Não foi testada generalização entre escritores ou reconhecimento de frases; esses são limites reais do trabalho, e não funcionalidades comprovadas. Melhorias futuras incluem novas caligrafias, coleta das classes faltantes e maior exploração de hiperparâmetros.

## Referências
- COHEN, G. et al. *EMNIST: An Extension of MNIST to Handwritten Letters*. 2017. https://arxiv.org/abs/1702.05373
- NIST. *EMNIST Dataset*. https://www.nist.gov/itl/products-and-services/emnist-dataset
- SCIKIT-LEARN. *SVC; MLPClassifier; GroupKFold; GroupShuffleSplit; Metrics*. https://scikit-learn.org/stable/
- NEUROWRITE-PT. *Repositório e execução reproduzível de treinamento*. https://github.com/Carlosfrancog/NeuroWrite-PT e https://github.com/Carlosfrancog/NeuroWrite-PT/actions/runs/42309016187
- **Dados manuscritos próprios**: folhas de Isadora, fotografadas em novembro de 2026, base segmentada e revisada de acordo com os arquivos do projeto.